In [1]:
!pip install ucimlrepo tensorflow --quiet


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler
from ucimlrepo import fetch_ucirepo
from sklearn.metrics import mean_squared_error


In [7]:
def autoencoder_reconstruction_mse(X, bottleneck_sizes, epochs=50, batch_size=32, random_state=42):
    """
    Train autoencoder for each bottleneck size and compute reconstruction MSE.

    Parameters:
        X (numpy.ndarray): Input data (samples x features)
        bottleneck_sizes (list): List of bottleneck dimensions
        epochs (int): Training epochs
        batch_size (int): Batch size
        random_state (int): Random seed

    Returns:
        pd.DataFrame: Dataset with bottleneck size and reconstruction MSE
    """
    np.random.seed(random_state)
    tf.random.set_seed(random_state)

    # Standardize data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    results = []

    for k in bottleneck_sizes:
        input_dim = X_scaled.shape[1]

        # Define autoencoder
        input_layer = layers.Input(shape=(input_dim,))
        encoded = layers.Dense(128, activation='relu')(input_layer)
        encoded = layers.Dense(k, activation='relu')(encoded)
        decoded = layers.Dense(128, activation='relu')(encoded)
        decoded = layers.Dense(input_dim, activation='linear')(decoded)

        autoencoder = models.Model(inputs=input_layer, outputs=decoded)
        autoencoder.compile(optimizer='adam', loss='mse')

        # Train
        autoencoder.fit(X_scaled, X_scaled, epochs=epochs, batch_size=batch_size, verbose=0,
                        validation_split=0.2)

        # Reconstruction
        X_reconstructed = autoencoder.predict(X_scaled, verbose=0)
        mse = mean_squared_error(X_scaled, X_reconstructed)

        results.append({
            'Bottleneck Size': k,
            'Reconstruction MSE': mse
        })

    return pd.DataFrame(results)


In [8]:
# Reduction levels for table
bottlenecks_wine = [1, 2, 5, 8, 9]        # Wine: 11 features
bottlenecks_breast = [3, 7, 15, 22, 27]  # Breast Cancer: 30 features
bottlenecks_mnist = [78, 196, 392, 588, 705]  # MNIST: 784 features


In [9]:
# Load Wine dataset
wine_data = fetch_ucirepo(id=186)
X_wine = wine_data.data.features

# Apply Autoencoder
wine_ae_results = autoencoder_reconstruction_mse(X_wine, bottlenecks_wine, epochs=50)
wine_ae_results['Dataset'] = 'Wine'
wine_ae_results


,Bottleneck Size,Reconstruction MSE,Dataset
0,1,0.539045,Wine
1,2,0.328510,Wine
2,5,0.095658,Wine
3,8,0.037575,Wine
4,9,0.009526,Wine


In [10]:
# Load Breast Cancer dataset
breast_data = fetch_ucirepo(id=17)
X_breast = breast_data.data.features

# Apply Autoencoder
breast_ae_results = autoencoder_reconstruction_mse(X_breast, bottlenecks_breast, epochs=50)
breast_ae_results['Dataset'] = 'Breast Cancer'
breast_ae_results


,Bottleneck Size,Reconstruction MSE,Dataset
0,3,0.232342,Breast Cancer
1,7,0.070641,Breast Cancer
2,15,0.020962,Breast Cancer
3,22,0.018385,Breast Cancer
4,27,0.017399,Breast Cancer


In [11]:
# Load MNIST dataset
(X_train, _), (_, _) = tf.keras.datasets.mnist.load_data()
X_mnist = X_train.reshape(X_train.shape[0], -1)  # Flatten to (60000, 784)

# Apply Autoencoder
mnist_ae_results = autoencoder_reconstruction_mse(X_mnist, bottlenecks_mnist, epochs=50)
mnist_ae_results['Dataset'] = 'MNIST'
mnist_ae_results


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


,Bottleneck Size,Reconstruction MSE,Dataset
0,78,0.300152,MNIST
1,196,0.262578,MNIST
2,392,0.277826,MNIST
3,588,0.287841,MNIST
4,705,0.306382,MNIST


In [12]:
# Combining all Autoencoder results
all_ae_results = pd.concat([wine_ae_results, breast_ae_results, mnist_ae_results], ignore_index=True)
all_ae_results


,Bottleneck Size,Reconstruction MSE,Dataset
0,1,0.539045,Wine
1,2,0.328510,Wine
2,5,0.095658,Wine
3,8,0.037575,Wine
4,9,0.009526,Wine
5,3,0.232342,Breast Cancer
6,7,0.070641,Breast Cancer
7,15,0.020962,Breast Cancer
8,22,0.018385,Breast Cancer
9,27,0.017399,Breast Cancer
